In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_pickle('./full_data.pkl')
data = data.loc[data['age']>18]
print(data.shape[0])

In [ ]:
data_add = pd.read_pickle('./ini_entropy.pkl')
data_beh = pd.read_pickle('./user_beh.pkl')

In [ ]:
data.columns

In [ ]:
data_add.columns

In [ ]:
data_beh.columns

In [ ]:
data = data.drop(columns=['expose_num_ex_search1', 'attention_span1', 'watched_search1','top10avg'])
data = data.merge(data_add, on='user_id')
data = data.merge(data_beh, on='user_id')

In [ ]:
def age_range_mapping(x):
    if 18<=x<=23:
        return '18-23'
    elif 24<=x<=30:
        return '24-30'
    elif 31<=x<=40:
        return '31-40'
    elif 41<=x<=50:
        return '41-50'
    elif 51<=x<=60:
        return '51-60'
    elif x>=61:
        return '60+'
    else:
        return None
    
def get_outliers(data, multiplier=1.5):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile).
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)

    # Calculate the interquartile range.
    IQR = Q3 - Q1

    # Define the lower and upper outlier bounds.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    return lower_bound, upper_bound


In [ ]:
data['age_range'] = data['age'].transform(lambda x: age_range_mapping(x))
data['activeness'] = data.apply(lambda x: x['collect1']+x['forward1']+x['comment1'], axis=1)

data = data.dropna()
data = data.loc[data['gender']!='UNKNOWN']
data = data.loc[data['fre_community_level_eng']!='unknown']
data = data.loc[data['fre_city_level_eng']!='unknown']
data = data.loc[data['mod_price_range']!='UNKNOWN']
data = data.loc[data['mod_price_range']!='UNKNOWN']
print(data.shape)


data_orig = data.copy()

s0_min, s0_max = get_outliers(data_orig['watch_entropy_1st'])
print('s0', s0_min, s0_max)
data  = data.loc[data['watch_entropy_1st']>=s0_min]
data  = data.loc[data['watch_entropy_1st']<=s0_max]

activeness_min, activeness_max = get_outliers(data_orig['activeness'])
print('activeness', activeness_min, activeness_max)
data  = data.loc[data['activeness']>=activeness_min]
data  = data.loc[data['activeness']<=activeness_max]


search_min, search_max = get_outliers(data_orig['watched_search1'])
print('search', search_min, search_max)
data  = data.loc[data['watched_search1']>=search_min]
data  = data.loc[data['watched_search1']<=search_max]


attention_min, attention_max = get_outliers(data_orig['attention_span1'])
print('attention', attention_min, attention_max)
data  = data.loc[data['attention_span1']>=attention_min]
data  = data.loc[data['attention_span1']<=attention_max]


top10avg_min,top10avg_max = get_outliers(data_orig['top10avg'])
print('top10avg', top10avg_min,top10avg_max)
data  = data.loc[data['top10avg']>=top10avg_min]
data  = data.loc[data['top10avg']<=top10avg_max]


In [ ]:
data_orig =[]
data.shape

In [ ]:
data = data[['user_id','m_ratio','gender', 'age' ,'age_range','fre_city_level_eng', 'fre_community_level_eng', 'mod_price_range',
            'activeness', 'watched_search1', 'attention_span1', 'top10avg', 'entertainment_ratio',
            'knowledge_ratio', 'news_ratio','watch_entropy_1st']]


In [ ]:
from scipy.stats import t,sem,norm
data_std = np.std(data['m_ratio'])
data_avg = np.mean(data['m_ratio'])

# Tail probability alpha.
alpha = 0.1

# Calculate the normal quantile.
z_value = norm.ppf(alpha/2)

# Print the result.
print("Entropy bounds:", 1-z_value*data_std, 1+z_value*data_std)
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<1+z_value*data_std else 0)

In [ ]:
population_list = []
gender_bias = []
age_bias = []
income_bias = []
city_bias = []

for c in criteria_list:
    data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c else 0)
    population_list.append(data['indicator'].mean())
    fig, axes = plt.subplots(1,4,figsize=(25, 4))
    sns.barplot(data, x='gender', y='indicator', order=['M', 'F'], ax=axes[0])
    sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-49','50+'], ax=axes[1])
    sns.barplot(data, x='mod_price_range', y='indicator', order=['1-1000','1000-2000','2000-4000','4000-5000','5000+'], ax=axes[2])
    sns.barplot(data, x='fre_community_level_eng', y='indicator', order=['City','County','Countryside'], ax=axes[3])

    axes[0].set_xlabel('Gender')
    axes[1].set_xlabel('Age')
    axes[2].set_xlabel('Phone Price')
    axes[3].set_xlabel('City Type')

    axes[0].set_ylabel('Propensity for Deep ICs')
    axes[1].set_ylabel('Propensity for Deep ICs')
    axes[2].set_ylabel('Propensity for Deep ICs')
    axes[3].set_ylabel('Propensity for Deep ICs')

    plt.suptitle(c)

    gender_tmp = data.groupby('gender')['indicator'].mean().reset_index()
    age_tmp = data.groupby('age_range')['indicator'].mean().reset_index()
    income_tmp = data.groupby('mod_price_range')['indicator'].mean().reset_index()
    city_tmp = data.groupby('fre_community_level_eng')['indicator'].mean().reset_index()

    gender_bias.append((gender_tmp.loc[gender_tmp['gender']=='F']['indicator'].to_numpy()[0]-gender_tmp.loc[gender_tmp['gender']=='M']['indicator'].to_numpy()[0])/data['indicator'].mean())
    age_bias.append((age_tmp.loc[age_tmp['age_range']=='50+']['indicator'].to_numpy()[0]-age_tmp.loc[age_tmp['age_range']=='18-23']['indicator'].to_numpy()[0])/data['indicator'].mean())
    income_bias.append((income_tmp.loc[income_tmp['mod_price_range']=='1-1000']['indicator'].to_numpy()[0]-income_tmp.loc[income_tmp['mod_price_range']=='5000+']['indicator'].to_numpy()[0])/data['indicator'].mean())   
    city_bias.append((city_tmp.loc[city_tmp['fre_community_level_eng']=='Countryside']['indicator'].to_numpy()[0]-city_tmp.loc[city_tmp['fre_community_level_eng']=='City']['indicator'].to_numpy()[0])/data['indicator'].mean())

In [ ]:
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'])
plt.ylabel('Propensity for Deep ICs')
plt.xlabel('Age')

In [ ]:
q_num1 = 10
q_num2 = 10
data['search_per_hour_q'] = pd.qcut(data['watched_search1'], q_num1, labels=False)
data['activeness_q'] = pd.qcut(data['activeness'], q_num1, labels=False)
data['attention_span_q'] = pd.qcut(data['attention_span1'], q_num2, labels=False)
data['top10avg_q'] = pd.qcut(data['top10avg'], q_num2, labels=False)

#data_tmp['explore_q'] = data_tmp.apply(lambda x: int((x['search_per_hour_q']+x['explored_per_hour_q'])/2), axis=1)
data['explore_q'] = data.apply(lambda x: int((x['search_per_hour_q']+x['activeness_q'])/2), axis=1)
data['exploit_q'] = data.apply(lambda x: int((x['attention_span_q']+x['top10avg_q'])/2), axis=1)


In [ ]:
data_groupss = data.groupby(['explore_q','exploit_q']).agg({'indicator':['sum','mean','count']}).reset_index()

In [ ]:
mat = np.zeros((q_num1,q_num2))
mat_cnt = np.zeros((q_num1,q_num2))
mat_in = np.zeros((q_num1,q_num2))
mat_out = np.zeros((q_num1,q_num2))

for i in range(q_num1):
    for j in range(q_num2):
        mat[i,j] = None if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['mean']
        mat_cnt[i,j] = 0 if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['count']
        mat_in[i,j] = 0 if data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator'].shape[0]==0 else data_groupss.loc[(data_groupss['explore_q']==i)&(data_groupss['exploit_q']==j)]['indicator']['sum']
        mat_out[i,j] = mat_cnt[i,j]-mat_in[i,j]

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(mat, center=0.205, vmin=0, vmax=0.41, cmap=sns.color_palette("vlag_r", as_cmap=True)).invert_yaxis()
plt.xticks(np.arange(0,10)+0.5,['~{}%'.format(i*10) for i in np.arange(1,11)])
plt.yticks(np.arange(0,10)+0.5,['~{}%'.format(i*10) for i in np.arange(1,11)])
plt.ylabel('Degree of Activeness')
plt.xlabel('Degree of Preferential Selection')

In [ ]:
plt.figure()
sns.barplot(data, x= 'search_per_hour_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'activeness_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xlabel('Quantile of Explored Videos per Hour')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'attention_span_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
plt.figure()
sns.barplot(data, x= 'top10avg_q', y='indicator')
plt.ylabel('Ratio of Deep ICs')
plt.xticks(np.arange(0,10),['{}%'.format(i*10) for i in np.arange(1,11)])

In [ ]:
sns.pointplot(data, x='explore_q' , y='indicator', hue='exploit_q')

In [ ]:
data['indicator_str'] = data['indicator'].transform(lambda x: 'Deep ICs' if x==1 else 'Others')
sns.barplot(data, x='age_range', y='watched_search1', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str')
plt.xlabel('Age')
plt.ylabel('# of search per hour')
plt.legend(title='State')

In [ ]:
sns.barplot(data, x='age_range', y='activeness', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str')
plt.xlabel('Age')
plt.ylabel('# of actions per hour')
plt.legend(title='State')

In [ ]:
sns.barplot(data, x='age_range', y='attention_span1', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str')
plt.xlabel('Age')
plt.ylabel('Average watching time on consumed videos')
plt.legend(title='State')

In [ ]:
sns.barplot(data, x='age_range', y='top10avg', order=['18-23','24-30','31-40','41-50','51-60','60+'], hue='indicator_str')
plt.xlabel('Age')
plt.ylabel('Average watching time on top 10% videos')
plt.legend(title='State')

--- Save a backup ---

In [ ]:
def age_range_conti(x):
    if x=='18-23':
        return 0
    elif x=='24-30':
        return 1
    elif x=='31-40':
        return 2
    elif x=='41-50':
        return 3
    elif x=='51-60':
        return 4
    elif x == '60+':
        return 5
    else:
        return None
def price_conti(x):
    if x == '1-1000':
        return 0
    elif x == '1000-2000':
        return 1
    elif x == '2000-4000':
        return 2
    elif x == '4000-5000':
        return 3
    elif x == '5000+':
        return 4
    else:
        return None

In [ ]:
data['age_conti'] = data['age_range'].transform(lambda x: age_range_conti(x))
data['price_conti'] = data['mod_price_range'].transform(lambda x: price_conti(x))

In [ ]:
data.to_csv('./cleaned_data_v2.csv')

In [ ]:
data['user_id'].to_pickle('./cleaned_user_id.pkl')

In [ ]:
data['user_id'].to_csv('./cleaned_user_id.csv', index=None)